# Post Race Evaluation

Input
- constructor and driver prediction tables
- hist driver and constructor points
- team config table (not yet created)


Processing
- log prediction absolute errors
- log best team (using teamOptimizer notebook/script)

Output
- show graph of trends of best team as well as my error rate

### Libraries

In [549]:
from pathlib import Path
import pandas as pd
import numpy as np
import os

import sys
sys.path.append("/Users/jackguptill/Library/CloudStorage/OneDrive-Personal/Code/F1FantasyProject")

from teamOptimizer import optimize_team

In [550]:
import os
os.getcwd()

'/Users/jackguptill/Library/CloudStorage/OneDrive-Personal/Code/F1FantasyProject/notebooks/evaluation'

In [551]:
working_directory = Path.cwd().parent.parent
print(working_directory)

/Users/jackguptill/Library/CloudStorage/OneDrive-Personal/Code/F1FantasyProject


### Data Import

In [552]:
driver_path = working_directory / "data" / "predictions" / "drivers" / "driver_predictions_2026.csv"
con_path = working_directory / "data" / "predictions" / "constructors" / "constructor_preds_2026.csv"

driver_race_results_path = working_directory / "data" / "clean" / "driver_points.csv"
constructor_race_results_path = working_directory / "data" / "clean" / "constructor_points.csv"

team_path = working_directory / "data" / "myTeam" / "teamConfig.csv"

driver_preds = pd.read_csv(driver_path).drop(columns=['Unnamed: 0'])
constructor_preds = pd.read_csv(con_path)

driver_race_results = pd.read_csv(driver_race_results_path).drop(columns=['Unnamed: 0'])
con_race_results = pd.read_csv(constructor_race_results_path).drop(columns=['Unnamed: 0'])

team_config = pd.read_csv(team_path)

## Data Clean

- joining and cleaning tables to get a single table that has our prediction performance
- will eventually use this to create graphs on how we are doing

### Creating a "Log" of the performance of predictions and results
- will contain features such as absolute error rate

In [553]:
constructor_preds["asset_name"] = constructor_preds["constructor"]
constructor_preds = constructor_preds.drop(columns=["constructor"])

In [554]:
driver_preds["asset_name"] = driver_preds["driver"]
driver_preds = driver_preds.drop(columns=["driver", "constructor"])

In [555]:
constructor_preds["asset_type"] = "constructor"

In [556]:
driver_preds["asset_type"] = "driver"

In [557]:
constructor_preds.columns

Index(['Unnamed: 0', 'year', 'race_num', 'price', 'predicted_points',
       'asset_name', 'asset_type'],
      dtype='object')

In [558]:
driver_preds.columns

Index(['Unnamed: 0.1', 'year', 'race_num', 'price', 'predicted_points',
       'asset_name', 'asset_type'],
      dtype='object')

In [559]:
predictions = pd.concat([constructor_preds, driver_preds], axis=0, ignore_index = True)
predictions["asset_name"] = predictions['asset_name'].str.upper()

In [560]:
predictions.head(100)

,Unnamed: 0,year,race_num,price,predicted_points,asset_name,asset_type,Unnamed: 0.1
0,0.0,2026,1,28.9,53.242756,MCL,constructor,NaN
1,1.0,2026,1,29.3,51.584534,MER,constructor,NaN
2,2.0,2026,1,28.2,48.359364,RBR,constructor,NaN
3,3.0,2026,1,23.3,40.582527,FER,constructor,NaN
4,4.0,2026,1,6.3,26.878194,RB,constructor,NaN
...,...,...,...,...,...,...,...,...
61,NaN,2026,1,6.4,5.940938,BOR,driver,39.0
62,NaN,2026,1,6.2,8.773124,COL,driver,40.0
63,NaN,2026,1,6.2,2.862940,LIN,driver,41.0
64,NaN,2026,1,6.0,3.668579,PER,driver,42.0


## Left Join Results to the predictions

In [561]:
driver_race_results = driver_race_results[['year', 'race', 'driver', 'points']]
con_race_results = con_race_results[["year", "race", "constructor", "points"]]
predictions = predictions.merge(driver_race_results, how = "left", left_on=["year", "race_num", "asset_name"], right_on= ["year", "race", "driver"]).drop(columns=["driver", "race"])
predictions = predictions.merge(con_race_results, how = "left", left_on=["year", "race_num", "asset_name"], right_on= ["year", "race", "constructor"]).drop(columns=["constructor", "race"])

#collapses points into a single col after the two joins
predictions["points"] = predictions["points_x"].combine_first(predictions["points_y"])
#dropping dups or unnamed cols
predictions = predictions.drop(columns=["points_x", "points_y","Unnamed: 0", "Unnamed: 0.1"])

predictions.head(30)

,year,race_num,price,predicted_points,asset_name,asset_type,points
0,2026,1,28.9,53.242756,MCL,constructor,19.0
1,2026,1,29.3,51.584534,MER,constructor,96.0
2,2026,1,28.2,48.359364,RBR,constructor,42.0
3,2026,1,23.3,40.582527,FER,constructor,69.0
4,2026,1,6.3,26.878194,RB,constructor,35.0
5,2026,1,10.3,22.922543,AMR,constructor,-38.0
6,2026,1,12.0,22.280189,WIL,constructor,20.0
7,2026,1,7.4,19.857089,HAS,constructor,34.0
8,2026,1,12.5,16.529915,ALP,constructor,22.0
9,2026,1,6.6,10.462140,AUD,constructor,0.0


#### creating a clean copy for visualizations and metrics

In [562]:
performance = predictions.copy()

## Getting Our Overall Mean Absolute Error of Our Predictions

In [563]:
performance["absolute_error"] = abs(performance["points"] - performance['predicted_points'])
np.mean(performance["absolute_error"])

np.float64(15.238592298534046)

In [564]:
#FIXME: make a current week or past week MAE

## What was the Optimal Team Last Week?

In [585]:
#driver table for input
latest_row = performance.sort_values(["year", "race_num"]).iloc[-1]
year = latest_row["year"]
race = latest_row["race_num"]

last_week_driver_results = driver_race_results[(driver_race_results["year"] == year) & (driver_race_results["race"] == race)].reset_index(drop=True)
last_week_driver_results["race_num"] = last_week_driver_results["race"]
last_week_driver_results = last_week_driver_results.drop(columns=["race"])
last_week_driver_results.head()

,year,driver,points,race_num
0,2026,VER,50.0,1
1,2026,RUS,39.0,1
2,2026,NOR,21.0,1
3,2026,PIA,-14.0,1
4,2026,ANT,32.0,1


In [584]:
latest_row = performance.sort_values(["year", "race_num"]).iloc[-1]
year = latest_row["year"]
race = latest_row["race_num"]

last_week_con_race_results = con_race_results[(con_race_results["year"] == year) & (con_race_results["race"] == race)].reset_index(drop=True)
last_week_con_race_results["race_num"] = last_week_con_race_results["race"]
last_week_con_race_results = last_week_con_race_results.drop(columns=["race"])
last_week_con_race_results.head()

,year,constructor,points,race_num
0,2026,MER,96.0,1
1,2026,MCL,19.0,1
2,2026,RBR,42.0,1
3,2026,FER,69.0,1
4,2026,ALP,22.0,1


In [586]:
drivers_sel, cons_sel, summary = optimize_team(
    budget = 100,
    drivers = last_week_driver_results,
    cons = last_week_con_race_results,
    points_col = "points",
    n_drivers = 5,
    n_constructors = 2,
    max_drivers_per_team = 2,
    use_drs = True,
    drs_multiplier = 2.0,
    solver_msg = False,
    require_driver_from_each_constructor = False,
    min_drivers_per_selected_constructor = 1
)

print(summary)
display(drivers_sel)
display(cons_sel)

TypeError: optimize_team() got an unexpected keyword argument 'drivers'